In [1]:
"""
Exp01C: Metric Sampling Distribution (Rolling Windows)

Goal:
- Treat backtest metrics (Sharpe / Annual Return / MDD / Turnover) as random variables
  under finite-sample uncertainty.
- Produce an empirical distribution over rolling sub-samples (windows).

Setting:
- Daily data
- Long-only, single MA (already in your pipeline)
- Signal-only baseline (no vol targeting / no risk-off)
- COST_RATE typically set to 0 here (you already handled cost in Exp01A)

Method:
- Build rolling windows over the available trading dates
- For each window: run(cfg with START/END), collect summary metrics
- Analyze the distribution: quantiles + tail probabilities
"""

'\nExp01C: Metric Sampling Distribution (Rolling Windows)\n\nGoal:\n- Treat backtest metrics (Sharpe / Annual Return / MDD / Turnover) as random variables\n  under finite-sample uncertainty.\n- Produce an empirical distribution over rolling sub-samples (windows).\n\nSetting:\n- Daily data\n- Long-only, single MA (already in your pipeline)\n- Signal-only baseline (no vol targeting / no risk-off)\n- COST_RATE typically set to 0 here (you already handled cost in Exp01A)\n\nMethod:\n- Build rolling windows over the available trading dates\n- For each window: run(cfg with START/END), collect summary metrics\n- Analyze the distribution: quantiles + tail probabilities\n'

In [2]:
import os
os.chdir("..")
print(os.getcwd())

/Users/kim/Desktop/Quant-MA


In [3]:
from copy import deepcopy
import pandas as pd

from config import Config
from runner import run

In [4]:
### 3 年窗 + 每年滚动（更接近“rolling”但仍很简单）

subsamples = {
    "2015-2017": ("2015-01-02", "2017-12-29"),
    "2016-2018": ("2016-01-04", "2018-12-31"),
    "2017-2019": ("2017-01-03", "2019-12-31"),
    "2018-2020": ("2018-01-02", "2020-12-31"),
    "2019-2021": ("2019-01-02", "2021-12-31"),
    "2020-2022": ("2020-01-02", "2022-12-30"),
    "2021-2023": ("2021-01-04", "2023-12-29"),
    "2022-2024": ("2022-01-03", "2024-12-31"),
    "2023-2025": ("2023-01-03", "2025-01-03"),
    "full":      ("2015-01-02", "2025-01-03"),
}

In [5]:
from dataclasses import replace
cfg_base = Config()
cfg_base = replace(cfg_base, MA_WINDOW=80, COST_RATE=0)   # 选一个 Exp01 plateau 的中心80, COST_RATE=0

In [8]:
rows = []

for name, (start, end) in subsamples.items():
    cfg = deepcopy(cfg_base)
    cfg = replace(cfg, START = start, END = end) 

    _, s = run(cfg)

    s["subsample"] = name
    s["START"] = start
    s["END"] = end
    s["MA_WINDOW"] = cfg.MA_WINDOW

    rows.append(s)

exp01c = pd.DataFrame(rows)
exp01c


/Users/kim/Desktop/Quant-MA/data/loaders.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(path, index_col=0, parse_dates=True)
/Users/kim/Desktop/Quant-MA/data/loaders.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  idx = pd.to_datetime(df.index, errors="coerce")
/Users/kim/Desktop/Quant-MA/data/loaders.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(path, index_col=0, parse_dates=True)
/Users/kim/Desktop/Quant-MA/data/loaders.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back

,Annual Return,Max Drawdown,Sharpe,Total Turnover,N_obs,subsample,START,END,MA_WINDOW
0,0.077766,-0.100623,0.925449,26.0,675,2015-2017,2015-01-02,2017-12-29,80
1,0.077521,-0.141527,0.878430,17.0,674,2016-2018,2016-01-04,2018-12-31,80
2,0.055721,-0.141527,0.629130,30.0,674,2017-2019,2017-01-03,2019-12-31,80
3,0.094346,-0.140623,0.817574,29.0,676,2018-2020,2018-01-02,2020-12-31,80
4,0.129909,-0.140623,0.978216,32.0,677,2019-2021,2019-01-02,2021-12-31,80
5,0.097406,-0.142352,0.758754,31.0,676,2020-2022,2020-01-02,2022-12-30,80
6,0.042728,-0.169437,0.433048,36.0,673,2021-2023,2021-01-04,2023-12-29,80
7,0.093018,-0.158261,0.819498,31.0,673,2022-2024,2022-01-03,2024-12-31,80
8,0.204541,-0.084056,1.723614,10.0,422,2023-2025,2023-01-03,2025-01-03,80
9,0.081822,-0.169437,0.765314,108.0,2436,full,2015-01-02,2025-01-03,80


In [10]:
def summarize_metric(df, col):
    x = pd.to_numeric(df[col], errors="coerce").dropna()
    return {
        "median": x.median(),
        "p25": x.quantile(0.25),
        "p75": x.quantile(0.75),
        "min": x.min(),
        "max": x.max(),
        "P(<0)": (x < 0).mean(),
    }

pd.DataFrame({
    "Sharpe": summarize_metric(exp01c, "Sharpe"),
    "Annual Return": summarize_metric(exp01c, "Annual Return"),
    "Max Drawdown": summarize_metric(exp01c, "Max Drawdown"),
    "Total Turnover": summarize_metric(exp01c, "Total Turnover")
}).T


,median,p25,p75,min,max,P(<0)
Sharpe,0.818536,0.760394,0.913694,0.433048,1.723614,0.0
Annual Return,0.087420,0.077583,0.096641,0.042728,0.204541,0.0
Max Drawdown,-0.141527,-0.154284,-0.140623,-0.169437,-0.084056,1.0
Total Turnover,30.500000,26.750000,31.750000,10.000000,108.000000,0.0
